# MLP search on Kaggle
Optuna TPE + Hyperband, 100 trials (`conf/hyperparameters/search.yaml`), about 6 hours on a T4. The study is saved in
`study.log`, so a search cut by Kaggle's 12-hour limit continues in the next session.
- Settings: GPU, Internet on. Input: the dataset `sandrokhizanishvili/hi-small-gnn`. Run with Save Version -> Save & Run All.
- Search: `N_TRIALS = 100`.
- Next session (search cut short, or the final after the search is checked): add the previous output as input
  (Add Input -> Notebook Output Files), set `PREVIOUS`; for the final also `RUN_FINAL = True`.
- Output: `/kaggle/working/outputs/tuning/mlp/mlp_readout-full/` (`study.log`, `trials.csv`, `tuning_curves.png`, `recipe.json`).

In [1]:
N_TRIALS = 100       # the session stops when the study has this many trials
PREVIOUS = '/kaggle/input/notebooks/sandrokh/gnn-for-aml-mlp-search/outputs'   # the previous session's outputs folder, '' in the first session
RUN_FINAL = True     # True only after the search is checked

In [2]:
# Pinned torch / PyG stack for the Kaggle GPU.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2 optuna==5.0.0

Found existing installation: torch 2.11.0+cu130
Uninstalling torch-2.11.0+cu130:
  Successfully uninstalled torch-2.11.0+cu130
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 366.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 246.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 287.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 371.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 477.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 441.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [3]:
# The code from GitHub, then the previous session's study (recipe.json is written again at the end).
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml
!ls /kaggle/input
if PREVIOUS:
    !mkdir -p /kaggle/working/outputs && cp -rn {PREVIOUS}/tuning /kaggle/working/outputs/ && rm -f /kaggle/working/outputs/tuning/*/*/recipe.json && ls /kaggle/working/outputs/tuning/*/*

Cloning into '/kaggle/working/gnn_for_aml'...
remote: Enumerating objects: 318, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (214/214), done.
remote: Total 318 (delta 168), reused 247 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (318/318), 577.37 KiB | 1.27 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/kaggle/working/gnn_for_aml
datasets  notebooks
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
config.yaml  study.log	trials.csv  tuning_curves.png


In [4]:
# Environment check.
!python run.py mode=check machine=kaggle

python 3.13.16 | torch 2.8.0+cu126 | CUDA build 12.6 | device cuda
  torch_geometric 2.8.1
  torch_scatter 2.1.2+pt28cu126
  torch_sparse 0.6.18+pt28cu126
  lightgbm 4.6.0
  optuna 5.0.0
  hydra-core 1.3.2
  hydra-submitit-launcher not installed (only needed on Cineca)
  numpy 2.1.3
  scipy 1.16.3
  pandas 2.3.3
  scikit-learn 1.6.1
  matplotlib 3.10.0
  psutil 5.9.5
/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
  gin         relu  fixed settings | params   54,083 | loss 0.6594
  gin         prelu {'train_eps': False, 'mlp_width': 2} | params   70,854 | loss 0.7223
  pna         relu  fixed settings | params  176,833 | loss 0.6665
  pna         prelu {'degree_scalers': False, 'towers': 4, 'divide_input': True} | params   54,278 | los

In [5]:
# The search; trials already in study.log are kept.
!python run.py tuning=mlp/search hyperparameters.search.n_trials={N_TRIALS} machine=kaggle

=== tune mlp_readout-full seed 42 -> /kaggle/working/outputs/tuning/mlp/mlp_readout-full
mode: tune
seed: 42
worker: 0
node_encoding: none
gfp_sheet: default
hyperparameters:
  search:
    n_trials: 100
    sampler_seed: 42
    startup_trials: 5
    reduction_factor: 3
    final_seeds:
    - 42
    - 43
    - 44
    - 45
    - 46
    first_check: 4
    start:
      hidden: 512
      depth: 3
      shape: funnel
      norm: batch
      dropout: 0.2
      lr: 0.001
      weight_decay: 1.0e-05
      pos_weight: 8.0
      batch_size: 8192
      activation: relu
    space:
      hidden:
      - 64
      - 128
      - 256
      - 512
      depth:
      - 2
      - 3
      - 4
      - 5
      shape:
      - flat
      - funnel
      - resnet
      norm:
      - none
      - batch
      - layer
      dropout:
      - 0.2
      - 0.3
      - 0.5
      - 0.7
      lr:
      - 0.0005
      - 0.001
      - 0.003
      - 0.005
      - 0.01
      weight_decay:
      - 0.0
      - 1.0e-05
      - 5.0

In [6]:
# Trials so far, best first.
import glob, optuna, pandas as pd
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend, JournalFileOpenLock
path = glob.glob('/kaggle/working/outputs/tuning/mlp/*/study.log')[0]
study = optuna.load_study(study_name=os.path.basename(os.path.dirname(path)),
                          storage=JournalStorage(JournalFileBackend(path, lock_obj=JournalFileOpenLock(path))))
trials = study.trials_dataframe()
print(trials['state'].value_counts().to_dict())
trials.sort_values('value', ascending=False).head(15)

{'PRUNED': 79, 'COMPLETE': 21}


,number,value,datetime_start,datetime_complete,duration,params_activation,params_batch_size,params_depth,params_dropout,params_hidden,...,user_attrs_start,user_attrs_threshold,user_attrs_val_f1,user_attrs_worker,system_attrs_completed_rung_0,system_attrs_completed_rung_1,system_attrs_completed_rung_2,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
80,80,0.480668,2026-11-22 14:01:17.884362,2026-11-22 14:05:31.590264,0 days 00:04:13.705902,relu,8192,3,0.3,64,...,NaN,0.219669,0.546176,0,0.475156,NaN,NaN,NaN,"{""activation"": ""relu"", ""batch_size"": 8192, ""de...",COMPLETE
69,69,0.479422,2026-11-22 13:39:41.121004,2026-11-22 13:43:51.422085,0 days 00:04:10.301081,relu,8192,3,0.3,64,...,NaN,0.229054,0.537731,0,0.471671,NaN,NaN,NaN,"{""activation"": ""relu"", ""batch_size"": 8192, ""de...",COMPLETE
67,67,0.478272,2026-11-22 13:35:11.534329,2026-11-22 13:39:12.021604,0 days 00:04:00.487275,relu,8192,2,0.3,64,...,NaN,0.452998,0.536145,0,0.439518,0.478272,NaN,NaN,"{""activation"": ""relu"", ""batch_size"": 8192, ""de...",COMPLETE
95,95,0.477872,2026-11-22 15:27:20.332144,2026-11-22 15:36:41.073038,0 days 00:09:20.740894,relu,2048,2,0.3,128,...,NaN,0.442109,0.551163,0,0.415350,0.458480,0.477872,NaN,"{""activation"": ""relu"", ""batch_size"": 2048, ""de...",COMPLETE
47,47,0.477555,2026-11-22 12:53:51.961257,2026-11-22 13:00:39.229161,0 days 00:06:47.267904,relu,8192,2,0.3,512,...,NaN,0.489831,0.538143,0,0.409736,0.452955,0.477555,NaN,"{""activation"": ""relu"", ""batch_size"": 8192, ""de...",COMPLETE
81,81,0.476930,2026-11-22 14:05:31.612602,2026-11-22 14:14:13.212601,0 days 00:08:41.599999,relu,2048,2,0.3,512,...,NaN,0.311888,0.545777,0,0.459920,0.476930,NaN,NaN,"{""activation"": ""relu"", ""batch_size"": 2048, ""de...",COMPLETE
92,92,0.476461,2026-11-22 14:50:31.157360,2026-11-22 15:11:57.753813,0 days 00:21:26.596453,relu,2048,2,0.3,512,...,NaN,0.295120,0.541618,0,0.428984,0.470186,0.476461,NaN,"{""activation"": ""relu"", ""batch_size"": 2048, ""de...",COMPLETE
63,63,0.476081,2026-11-22 13:28:42.520497,2026-11-22 13:31:54.431978,0 days 00:03:11.911481,relu,2048,2,0.3,512,...,NaN,NaN,NaN,0,0.429027,0.459442,0.476081,NaN,"{""activation"": ""relu"", ""batch_size"": 2048, ""de...",PRUNED
43,43,0.474742,2026-11-22 12:47:02.761427,2026-11-22 12:52:21.074785,0 days 00:05:18.313358,relu,8192,3,0.2,64,...,NaN,0.518175,0.531216,0,0.441977,0.461578,NaN,NaN,"{""activation"": ""relu"", ""batch_size"": 8192, ""de...",COMPLETE
94,94,0.474665,2026-11-22 15:19:57.106354,2026-11-22 15:27:20.284607,0 days 00:07:23.178253,swish,8192,2,0.3,512,...,NaN,NaN,NaN,0,0.409296,0.456898,0.474665,NaN,"{""activation"": ""swish"", ""batch_size"": 8192, ""d...",PRUNED


In [7]:
# Final: the winner refit on train + val, seeds 42-46; a finished seed is skipped.
if RUN_FINAL and glob.glob('/kaggle/working/outputs/tuning/mlp/*/recipe.json'):
    !python run.py -m tuning=mlp/final machine=kaggle

[2026-11-22 17:46:19,608][HYDRA] Launching 5 jobs locally
[2026-11-22 17:46:19,608][HYDRA] 	#0 : seed=42 tuning=mlp/final machine=kaggle
=== final mlp_readout-full seed 42 -> /kaggle/working/outputs/final/mlp/mlp_readout-full/seed-42
mode: final
seed: 42
worker: 0
node_encoding: none
gfp_sheet: default
hyperparameters:
  search:
    n_trials: 100
    sampler_seed: 42
    startup_trials: 5
    reduction_factor: 3
    final_seeds:
    - 42
    - 43
    - 44
    - 45
    - 46
    first_check: 4
    start:
      hidden: 512
      depth: 3
      shape: funnel
      norm: batch
      dropout: 0.2
      lr: 0.001
      weight_decay: 1.0e-05
      pos_weight: 8.0
      batch_size: 8192
      activation: relu
    space:
      hidden:
      - 64
      - 128
      - 256
      - 512
      depth:
      - 2
      - 3
      - 4
      - 5
      shape:
      - flat
      - funnel
      - resnet
      norm:
      - none
      - batch
      - layer
      dropout:
      - 0.2
      - 0.3
      - 0.5
     

In [8]:
# Small archive for download (no checkpoints, no predictions).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

-rw-r--r-- 1 root root 1.1M Nov 22 18:13 results_small.zip


## After the run
- Download `results_small.zip` from the Output tab and unzip it into `gnn_for_aml/outputs/`.
- Fewer than 100 trials: next session with `PREVIOUS` set.